# Avance 6 - Conclusiones Clave
## Prediccion de Metano Bovino mediante ML
**Equipo 48 - Tec de Monterrey | CRISP-ML(Q) | Junio 2026**

| Entregable | Resultado clave |
|---|---|
| **E3** Baseline Ridge | RMSE=0.7268, R2=0.9688 |
| **E4** MLP alternativo | RMSE=0.6555, R2=0.9746, -9.8% vs E3 |
| **E5** Stacking ensamble | RMSE=0.6117, R2=0.9779, -15.84% vs E3 |
| **E6** Conclusiones e implementacion | Este documento |


In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings, json
from pathlib import Path
warnings.filterwarnings('ignore')

REPO = Path.cwd().resolve()
if not (REPO / 'data').exists():
    REPO = REPO.parent

OUTPUT = REPO / 'data' / 'processed' / 'e6_outputs'
OUTPUT.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 130, 'axes.facecolor': '#1a1d27',
    'figure.facecolor': '#0f1117', 'text.color': '#eef0f7',
    'axes.labelcolor': '#eef0f7', 'xtick.color': '#8890a8',
    'ytick.color': '#8890a8', 'axes.edgecolor': '#2a2f45',
    'grid.color': '#2a2f45', 'grid.alpha': 0.5
})
print('Setup listo')


## 1. Resultados consolidados E3-E5

In [ ]:
resultados = {
    'Modelo':      ['Ridge-E3','MLP-E4','SVR-Lin-E4','BayesianRidge-E4','Stacking-E5','Voting-E5'],
    'RMSE':        [0.7268, 0.6555, 0.7130, 0.7268, 0.6117, 0.6485],
    'R2':          [0.9688, 0.9746, 0.9700, 0.9688, 0.9779, 0.9752],
    'MAPE_pct':    [3.77, 3.39, 3.62, 3.77, 3.06, 3.30],
    'Latencia_ms': [0.0, 4.4, 0.08, 0.07, 7.3, 6.7],
    'Delta_E3_pct':[0.0, -9.8, -1.9, 0.0, -15.84, -10.7],
    'Entrega':     ['E3','E4','E4','E4','E5','E5'],
}
df_res = pd.DataFrame(resultados)
print('RESULTADOS CONSOLIDADOS E3-E5')
print('='*70)
print(df_res.to_string(index=False))
winner = df_res.loc[df_res['RMSE'].idxmin()]
print(f'\nModelo final: {winner.Modelo}')
print(f'  RMSE={winner.RMSE}  R2={winner.R2}  Latencia={winner.Latencia_ms}ms  Delta={winner.Delta_E3_pct}%')
df_res.to_csv(OUTPUT / 'e6_resultados_consolidados.csv', index=False)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
modelos = df_res['Modelo'].values
rmses   = df_res['RMSE'].values
r2s     = df_res['R2'].values
colors = ['#00c896' if r==0.6117 else '#4b9dff' if r<0.7268 else '#8890a8' for r in rmses]

bars = axes[0].bar(range(len(modelos)), rmses, color=colors, edgecolor='none', width=0.65)
axes[0].axhline(0.7268, color='#ffc857', ls='--', lw=1.5, label='Baseline E3 (0.7268)')
axes[0].axhline(0.75,   color='#ff5757', ls=':',  lw=1.2, label='Objetivo max (0.75)')
for b, r in zip(bars, rmses):
    axes[0].text(b.get_x()+b.get_width()/2, r+0.008, f'{r:.4f}',
                 ha='center', va='bottom', fontsize=8, color='#eef0f7', fontweight='bold')
axes[0].set_xticks(range(len(modelos)))
axes[0].set_xticklabels(modelos, rotation=25, ha='right', fontsize=8)
axes[0].set_ylabel('RMSE (g CH4/kg leche)')
axes[0].set_title('Progresion RMSE - E3 a E5', fontweight='bold')
axes[0].legend(fontsize=9); axes[0].set_ylim(0, 0.85)

cols2 = ['#00c896' if r>0.977 else '#4b9dff' if r>0.969 else '#8890a8' for r in r2s]
bars2 = axes[1].bar(range(len(modelos)), r2s, color=cols2, edgecolor='none', width=0.65)
axes[1].axhline(0.95, color='#ff5757', ls=':', lw=1.2, label='Objetivo min (0.95)')
axes[1].set_ylim(0.90, 0.985)
for b, r in zip(bars2, r2s):
    axes[1].text(b.get_x()+b.get_width()/2, r+0.0003, f'{r:.4f}',
                 ha='center', va='bottom', fontsize=8, color='#eef0f7', fontweight='bold')
axes[1].set_xticks(range(len(modelos)))
axes[1].set_xticklabels(modelos, rotation=25, ha='right', fontsize=8)
axes[1].set_ylabel('R2'); axes[1].set_title('Progresion R2 - E3 a E5', fontweight='bold')
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT / 'e6_01_progresion_modelos.png', bbox_inches='tight')
print('Guardado: e6_01_progresion_modelos.png')


## 2. Objetivo 4.1 - Evaluacion Go/No-Go

In [ ]:
criterios = {
    'Criterio': [
        'RMSE < 0.75 g CH4/kg leche',
        'R2 > 0.95',
        'Latencia prediccion < 100 ms',
        'Sin data leakage (GroupKFold)',
        'Estabilidad CV inter-fold < 5%',
        'Drift documentado con mitigacion',
        'Significancia estadistica vs baseline (p < 0.05)',
        'Interpretabilidad SHAP disponible',
        'Criterios de negocio E1 satisfechos',
    ],
    'Objetivo': ['< 0.75','> 0.95','< 100ms','Validado','< 5%',
                 'Plan definido','p < 0.05','SHAP OK','Si'],
    'Resultado_E5': ['0.6117 OK','0.9779 OK','7.3ms OK','GroupKFold OK','0.9% OK',
                     'Retrain trimestral OK','p=0.0001 OK','KernelExplainer OK','Todos OK'],
    'Estado': ['GO']*9
}
df_gonogo = pd.DataFrame(criterios)
print('EVALUACION GO / NO-GO')
print('='*75)
print(df_gonogo.to_string(index=False))
print()
print('='*55)
print('  DECISION: PROCEDER A IMPLEMENTACION')
print('  Todos los criterios de exito estan cumplidos.')
print('  No retroceder a fases anteriores.')
print('='*55)
rmse_final = 0.6117
error_rel = rmse_final / (28.0 - 14.0) * 100
print(f'\nInterpretacion practica:')
print(f'  RMSE={rmse_final} g CH4/kg leche | Error relativo al rango: {error_rel:.1f}% | MAPE=3.06%')
print(f'  Si una vaca emite 20 g CH4/kg, el modelo predice entre {20-rmse_final:.1f} y {20+rmse_final:.1f} g CH4/kg (68% casos)')
df_gonogo.to_csv(OUTPUT / 'e6_gonogo.csv', index=False)


## 3. Objetivo 4.2 - Medidas y decisiones especificas

In [ ]:
acciones = {
    'Area': ['Modelo','Modelo','Modelo','Datos','Datos',
             'Operaciones','Operaciones','Stakeholders','Stakeholders','Stakeholders'],
    'Accion': [
        'Desplegar Stacking-E5 (pipeline_final_e5.pkl) como modelo de produccion',
        'Retrain trimestral con datos recientes (Q1-Q4)',
        'Monitoreo KS mensual - trigger automatico si p < 0.05',
        'Ampliar muestra Jersey y Pardo Suizo (sub-representadas)',
        'Integrar medicion real de lab en app para retroalimentacion continua',
        'Contenedor Docker para portabilidad en cualquier servidor',
        'CI/CD GitHub Actions: push -> build -> test -> deploy automatico',
        'Dashboard operativo para equipo de campo (URL Cloud)',
        'Alertas automaticas vacas nivel Alto (>25 g CH4/kg leche)',
        'Reporte mensual de hato para gerencia con KPIs y tendencia',
    ],
    'Prioridad': ['Alta','Alta','Alta','Media','Alta','Alta','Media','Alta','Media','Baja'],
    'Plazo': ['Inmediato','Trimestral','Mensual','6 meses','Inmediato',
              'Sprint 1','Sprint 2','Inmediato','Sprint 1','Trimestral']
}
df_acc = pd.DataFrame(acciones)
print('ACCIONES ACCIONABLES POR AREA')
print('='*75)
for area in df_acc['Area'].unique():
    sub = df_acc[df_acc['Area']==area]
    print(f'\n  [{area.upper()}]')
    for _, row in sub.iterrows():
        icon = '[!]' if row.Prioridad=='Alta' else '[~]'
        print(f'    {icon} {row.Accion}  |  {row.Plazo}')
df_acc.to_csv(OUTPUT / 'e6_acciones.csv', index=False)


## 4. Objetivo 4.3 - Plataforma cloud para produccion

In [ ]:
criterios_cloud = {
    'Criterio': ['Integracion Python/sklearn','Facilidad de uso','Escalabilidad',
                 'Costo proyecto pequeno-mediano','MLOps y monitoreo',
                 'Latencia prediccion','Soporte academico/creditos',
                 'Seguridad y cumplimiento','Comunidad y ecosistema'],
    'Peso':  [0.20, 0.15, 0.10, 0.20, 0.10, 0.10, 0.05, 0.05, 0.05],
    'AWS':   [9, 7, 10, 6, 9, 9, 7, 9, 10],
    'Azure': [9, 8,  9, 7, 9, 8, 9, 9,  8],
    'GCP':   [10,8,  9, 9, 9, 9, 9, 8,  8],
    'IBM':   [7, 6,  7, 5, 7, 7, 4, 9,  5],
}
df_sc = pd.DataFrame(criterios_cloud)
totals = {}
for p in ['AWS','Azure','GCP','IBM']:
    df_sc[f'S_{p}'] = df_sc[p] * df_sc['Peso']
    totals[p] = round(df_sc[f'S_{p}'].sum(), 2)

print('MATRIZ DE SCORING PONDERADA (0-10)')
print('='*70)
print(f'  {"Criterio":<38} {"Peso":>5}  {"AWS":>5}  {"Azure":>5}  {"GCP":>5}  {"IBM":>5}')
print('-'*70)
for _, row in df_sc.iterrows():
    print(f'  {row.Criterio:<38} {row.Peso:>5.2f}  {row.AWS:>5.0f}  {row.Azure:>5.0f}  {row.GCP:>5.0f}  {row.IBM:>5.0f}')
print('-'*70)
print(f'  {"TOTAL PONDERADO":<38} {"1.00":>5}  {totals["AWS"]:>5.2f}  {totals["Azure"]:>5.2f}  {totals["GCP"]:>5.2f}  {totals["IBM"]:>5.2f}')
ganador = max(totals, key=totals.get)
print(f'\nPROVEEDOR RECOMENDADO: {ganador} (score={totals[ganador]}/10.00)')
print()
print('JUSTIFICACION GCP:')
print('  1. Integracion nativa sklearn: Vertex AI acepta pipelines sklearn/joblib directamente')
print('  2. Costo Cloud Run serverless: ~$6/mes para 1000 predicciones/dia')
print('     vs SageMaker AWS: minimo ~$40/mes (instancia siempre activa)')
print('  3. $300 USD de credito inicial para nuevos proyectos (90 dias)')
print('  4. BigQuery gratuito para < 10 GB/mes - ideal para 73K registros')
print('  5. CI/CD automatico: GitHub push -> Cloud Build -> Cloud Run deploy')
print('  6. Firebase Hosting gratuito para el dashboard React/Vite')
df_sc.to_csv(OUTPUT / 'e6_scoring_cloud.csv', index=False)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
proveedores = ['AWS','Azure','GCP','IBM']
scores = [totals[p] for p in proveedores]
colores = ['#ffc857','#4b9dff','#00c896','#ff5757']

bars = axes[0].bar(proveedores, scores, color=colores, edgecolor='none', width=0.5)
axes[0].set_ylim(6.5, 9.5)
for b, s in zip(bars, scores):
    axes[0].text(b.get_x()+b.get_width()/2, s+0.05, f'{s:.2f}',
                 ha='center', va='bottom', fontsize=13, color='#eef0f7', fontweight='bold')
axes[0].set_title('Score total ponderado por proveedor', fontweight='bold')
axes[0].set_ylabel('Score (0-10)')

crit_short = [c[:20]+'...' if len(c)>20 else c for c in df_sc['Criterio']]
x = np.arange(len(crit_short)); w = 0.2
for i, (prov, col) in enumerate(zip(proveedores, colores)):
    axes[1].bar(x + i*w, df_sc[prov]*df_sc['Peso'], w, label=prov, color=col, alpha=0.85)
axes[1].set_xticks(x + 1.5*w)
axes[1].set_xticklabels(crit_short, rotation=35, ha='right', fontsize=8)
axes[1].set_title('Score ponderado por criterio', fontweight='bold')
axes[1].set_ylabel('Score x Peso'); axes[1].legend(fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT / 'e6_02_cloud_scoring.png', bbox_inches='tight')
print('Guardado: e6_02_cloud_scoring.png')


## 5. Arquitectura de produccion propuesta (GCP)

In [ ]:
print('''
ARQUITECTURA GCP - DIAGRAMA LOGICO
====================================

[CAMPO / SENSORES]
  Formulario app -> POST /api/registros -> prediccion Stacking -> alerta si Alto

[CAPA DATOS - GCP]
  Cloud Storage (CSV raw) -> Cloud SQL PostgreSQL (BD produccion)
  Backup automatico | Encriptado AES-256 | Acceso auditado

[CAPA ML - VERTEX AI]
  Vertex AI Pipelines:
    1. Feature engineering (28 features)
    2. Reentrenamiento trimestral (StackingRegressor)
    3. Validacion automatica RMSE < 0.75 -> aprueba o rechaza
    4. Model Registry -> versionado de modelos

[CAPA API - CLOUD RUN]
  POST /api/predict -> pipeline_final_e5.pkl -> JSON response
  Latencia p50: 7ms | p99: ~25ms | Auto-scale 0-100 instancias
  URL: https://metanoml-api-xxxx.run.app

[CAPA FRONTEND - FIREBASE HOSTING]
  React/Vite build -> CDN global -> HTTPS automatico
  URL: https://metanoml-equipo48.web.app

[OBSERVABILIDAD - CLOUD MONITORING]
  Alertas: RMSE > 0.75 -> email al equipo ML
  Dashboard: latencia, errores, predicciones/hora
  Logs: predicciones auditables, trazabilidad completa

COSTO ESTIMADO MENSUAL:
  Cloud Run API : ~$5 USD
  Cloud SQL     : ~$25 USD (db-f1-micro)
  Cloud Storage : ~$1 USD (< 10 GB)
  Firebase      : $0 USD (plan gratuito)
  TOTAL         : ~$31 USD/mes
''')


## 6. Margen de mejora futura

In [ ]:
mejoras = {
    'Propuesta': [
        'Lag features temporales (metano t-1, t-7, t-30)',
        'Ampliar muestra Jersey/Pardo Suizo a 200+ vacas/raza',
        'Transformer temporal (TFT) para series por vaca individual',
        'SHAP por prediccion individual integrado en dashboard',
        'A/B testing en produccion con shadow deployment',
        'Variables de pastura y estacionalidad DJF/MAM/JJA/SON',
    ],
    'Delta_RMSE_est': ['-5 a -10%','-3 a -8%','-8 a -15%','N/A','N/A','-3 a -6%'],
    'Complejidad': ['Media','Baja','Alta','Media','Alta','Media'],
    'Prioridad':   ['Alta','Alta','Baja','Alta','Media','Alta']
}
df_mej = pd.DataFrame(mejoras)
print('MARGEN DE MEJORA FUTURA')
print('='*80)
print(df_mej.to_string(index=False))
print()
print('Mayor impacto: lag features + datos balanceados -> RMSE ~0.50-0.55 estimado')
df_mej.to_csv(OUTPUT / 'e6_mejoras_futuras.csv', index=False)


## 7. Conclusiones finales

In [ ]:
print('''
CONCLUSIONES FINALES - EQUIPO 48 - TEC DE MONTERREY
=====================================================

1. VIABILIDAD CONFIRMADA
   Stacking-E5 cumple TODOS los criterios: RMSE=0.6117, R2=0.9779, lat=7.3ms.
   DECISION: Proceder a implementacion. No retroceder a fases anteriores.

2. PROGRESION SOLIDA
   E3 -> E4 -> E5: RMSE 0.7268 -> 0.6555 -> 0.6117 (-15.84% total)
   Stacking supero al mejor modelo individual (MLP) en 6.7%.

3. HALLAZGOS CLAVE DE ML
   - MLP bien calibrado desde E3; tuning N=50 no lo mejoro
   - SVR-Lin inesperadamente competitivo (RMSE=0.7130)
   - XGBoost no performo bien con estos datos tabulares
   - k-NN impractico para produccion (pkl=13MB, latencia=1086ms)
   - Ensamble captura complementariedad de modelos individuales

4. PLATAFORMA RECOMENDADA: GCP (Vertex AI + Cloud Run)
   Score ponderado mas alto (8.85/10).
   Costo: ~$31 USD/mes. Integracion sklearn nativa. $300 credito inicial.

5. IMPACTO PRACTICO
   Predecir metano sin equipos de laboratorio.
   Ahorro estimado: $200-500 USD/vaca/ano en analisis de gas.
   ROI esperado: < 6 meses en hato de 100+ vacas.
''')


In [ ]:
meta = {
    'equipo': 'Equipo 48 - Tec de Monterrey',
    'entregable': 'E6 - Conclusiones Clave',
    'modelo_final': 'Stacking (MLP + BayesianRidge + ElasticNet -> meta Ridge)',
    'rmse_final': 0.6117, 'r2_final': 0.9779, 'mape_final': 3.06,
    'latencia_ms': 7.3, 'mejora_vs_e3_pct': 15.84,
    'decision': 'GO - Proceder a implementacion',
    'plataforma': 'GCP (Vertex AI + Cloud Run)',
    'costo_mensual_usd': 31, 'criterios_cumplidos': '9 de 9',
}
with open(OUTPUT / 'e6_artifacts_meta.json', 'w') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)
print('E6 completo. Artefactos en:', str(OUTPUT))
for a in sorted(OUTPUT.glob('e6_*')):
    print(f'  {a.name} ({a.stat().st_size//1024+1} KB)')
